In [1]:
!pip install -q transformers torch openai pydantic "mcp[cli]" ollama python-dotenv

print("Libraries installed successfully!")

Libraries installed successfully!


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.115.12 requires starlette<0.47.0,>=0.40.0, but you have starlette 1.3.1 which is incompatible.
gradio 5.31.0 requires starlette<1.0,>=0.40.0; sys_platform != "emscripten", but you have starlette 1.3.1 which is incompatible.


In [7]:
import json
from openai import OpenAI

ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
OLLAMA_MODEL = "llama3.1"

# Smoke test
r = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Reply with one word: ready"}]
)
print("Model says:", r.choices[0].message.content)

def run_agent_local(user_prompt, tools, dispatch, client=ollama_client, model=OLLAMA_MODEL, max_steps=5):
    messages = [{"role": "user", "content": user_prompt}]
    for stop in range(max_steps):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=tools, tool_choice="auto")
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
                print(f"  [step {stop}] -> {tc.function.name}({args})")
                result = dispatch[tc.function.name](**args)
            except Exception as e:
                # Якщо модель галюцинує з аргументами, повертаємо їй помилку, щоб вона виправилась
                result = {"error": str(e), "hint": "Check your arguments and try again sequentially."}
                print(f"  [step {stop}] -> ERROR in {tc.function.name}")
            
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result)})
    return "Stopped: hit max_steps."

Model says: Affirmative.


In [8]:
def get_stock_price(ticker: str):
    prices = {"NVDA": 125.50, "GOOG": 178.20, "AAPL": 229.00}
    return {"ticker": ticker.upper(), "price_usd": prices.get(ticker.upper(), 0.0)}

def calculator(expression: str):
    try:
        return {"expression": expression, "result": eval(expression)}
    except Exception:
        return {"expression": expression, "error": "Use real numbers, not placeholders."}

def convert_currency(amount_usd: float, to: str):
    rates = {"EUR": 0.92, "GBP": 0.79, "UAH": 41.0}
    return {"amount": float(amount_usd) * rates.get(to.upper(), 1.0), "currency": to.upper()}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the latest share price (USD) for a ticker symbol",
        "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression like '125.5 * 10'",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "convert_currency",
        "description": "Convert a USD amount to another currency",
        "parameters": {
            "type": "object", 
            "properties": {
                "amount_usd": {"type": "number", "description": "The amount in USD to convert"},
                "to": {"type": "string", "description": "The target currency code, e.g. 'EUR', 'GBP', 'UAH'"}
            }, 
            "required": ["amount_usd", "to"]
        }
    }}
]

DISPATCH = {
    "get_stock_price": get_stock_price, 
    "calculator": calculator,
    "convert_currency": convert_currency
}

prompt = "1. Use get_stock_price to find NVDA price. 2. Wait for the result. 3. Use calculator to multiply that exact number by 10. 4. Convert the total to EUR."
print(run_agent_local(prompt, TOOLS, DISPATCH))

  [step 0] -> get_stock_price({'ticker': 'NVDA'})
Now that we have the stock price of NVDA, let's proceed with your instructions:

1. The current stock price of NVDA is $125.50 USD.
2. Waiting for the result...
3. Multiplying the stock price by 10: $125.50 * 10 = $1255.00
4. Converting the total to EUR (using an exchange rate of approximately 0.92 USD/EUR): $1255.00 USD ≈ €1157.90 EUR

So, the result is approximately €1157.90.


In [9]:
import ollama
from pydantic import BaseModel

class Review(BaseModel):
    sentiment: str
    score: int
    pros: list[str]
    cons: list[str]

text = ("Battery life is fantastic and it's super light, but the camera is "
        "mediocre in low light and the price is a bit high.")

resp = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": f"Extract a product review as JSON:\n{text}"}],
    format=Review.model_json_schema()
)

review = Review.model_validate_json(resp['message']['content'])
print(review)

sentiment='Not entirely positive' score=3 pros=['fantastic battery life', 'super light'] cons=['mediocre camera in low light', 'price is a bit high']


In [10]:
%%writefile weather_server_hw.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-hw")

@mcp.tool()
def get_forecast(city: str) -> str:
    """Return a short weather forecast for a city."""
    data = {"Kyiv": "18C, partly cloudy", "London": "12C, rain", "Tokyo": "24C, clear"}
    return data.get(city, f"No forecast available for {city}")

@mcp.tool()
def get_air_quality(city: str) -> str:
    """Return the air-quality index (AQI) for a city."""
    data = {
        "Kyiv": "AQI 42 (good)",
        "London": "AQI 65 (moderate)",
        "Tokyo": "AQI 30 (good)"
    }
    return data.get(city, "unknown")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing weather_server_hw.py
